In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [2]:
%%writefile content/stats.json

[
 {
  "key": "wOBA",
  "level": "進階",
  "group": "打擊",
  "full": "加權上壘率 Weighted On-Base Average",
  "one": "把保送、一安、二安、三安、全壘打依「對得分的實際貢獻」加權後算出的上壘率。",
  "why": "打擊率把一壘安打和全壘打都算成 1 支安打，上壘率又把保送和全壘打看得一樣重。wOBA 依每種結果實際能換到多少分給權重，一個數字就能看出打者整體的進攻價值。",
  "use": "比較打者整體攻擊力，找出打擊率普通但其實很會製造分數的人。",
  "formula": "\\text{wOBA}=\\frac{0.69\\,\\text{uBB}+0.72\\,\\text{HBP}+0.89\\,\\text{1B}+1.27\\,\\text{2B}+1.62\\,\\text{3B}+2.10\\,\\text{HR}}{\\text{AB}+\\text{BB}-\\text{IBB}+\\text{SF}+\\text{HBP}}\\times\\text{倍數}",
  "table": "batters",
  "column": "wOBA_calc",
  "min_col": "PA",
  "min": 200,
  "higher_better": true,
  "fmt": "{:.3f}"
 },
 {
  "key": "ISO",
  "level": "入門",
  "group": "打擊",
  "full": "純長打率 Isolated Power",
  "one": "長打率減掉打擊率，只留下「長打」的部分。",
  "why": "長打率包含一壘安打，高打擊率的短打型打者長打率也可能不差。ISO 把一安拿掉，只看打者打出長打的能力。",
  "use": "找出真正的長打者，或觀察打者是不是改變了打擊策略。",
  "formula": "\\text{ISO}=\\text{SLG}-\\text{AVG}=\\frac{\\text{2B}+2\\times\\text{3B}+3\\times\\text{HR}}{\\text{AB}}",
  "table": "batters",
  "column": "ISO_calc",
  "min_col": "PA",
  "min": 200,
  "higher_better": true,
  "fmt": "{:.3f}"
 },
 {
  "key": "BABIP",
  "level": "進階",
  "group": "打擊",
  "full": "場內球安打率 Batting Average on Balls In Play",
  "one": "只看打進場內（不含全壘打、三振）的球，有多少比例變成安打。",
  "why": "球打進場內之後，落點有很大的運氣成分。如果一位打者的 BABIP 遠高於自己的生涯水準，通常代表運氣不錯，之後很可能回落。",
  "use": "判斷一段時間的好成績（或壞成績）能不能持續。要跟球員自己的過去比，沒有絕對的好壞。",
  "formula": "\\text{BABIP}=\\frac{\\text{H}-\\text{HR}}{\\text{AB}-\\text{SO}-\\text{HR}+\\text{SF}}",
  "table": "batters",
  "column": "BABIP_calc",
  "min_col": "PA",
  "min": 200,
  "higher_better": true,
  "fmt": "{:.3f}"
 },
 {
  "key": "FIP",
  "level": "進階",
  "group": "投球",
  "full": "投手獨立防禦率 Fielding Independent Pitching",
  "one": "只用投手自己能控制的三件事（三振、保送、全壘打）估算出來的防禦率。",
  "why": "防禦率會受到守備和運氣影響。FIP 排除這些因素，更能反映投手本身的能力，也比 ERA 更能預測未來表現。",
  "use": "比較 ERA 和 FIP 的差距：ERA 遠高於 FIP，代表運氣差或被守備拖累。",
  "formula": "\\text{FIP}=\\frac{13\\,\\text{HR}+3\\,\\text{BB}-2\\,\\text{SO}}{\\text{IP}}+\\text{常數}",
  "table": "pitchers",
  "column": "FIP_calc",
  "min_col": "IP",
  "min": 40,
  "higher_better": false,
  "fmt": "{:.2f}"
 }
]

Writing content/stats.json


In [5]:
%%writefile pages/academy.py
import json
from pathlib import Path

import plotly.graph_objects as go
import streamlit as st

from src import metrics, ui
from src.data import load
ui.setup('數據學院')
teams,batters,pitchers,const,lg=load()
STATS=json.loads((Path(__file__).resolve().parent.parent/"content" / "stats.json").read_text(encoding='utf-8'))
TABLES = {"batters": batters, "pitchers": pitchers}

levels = ["入門", "進階", "高階"]
nav , main=st.columns([1,4])
with nav:
    key=st.radio('選擇數據',[s['key'] for s in STATS],format_func=lambda k: "{0[group]}｜{0[key]}（{0[level]}）".format(
                       next(s for s in STATS if s["key"] == k)))
s=next( x for x in STATS if x['key']==key)
df=TABLES[s['table']]
df=df[df[s['min_col']]>=s['min']]
col = s["column"]
fmt = ui.f3 if s["fmt"] == "{:.3f}" else s["fmt"].format   # 三位小數用棒球習慣：.320

with main:
    st.markdown(" → ".join(f"**{l}**" if l == s["level"] else l for l in levels))
    st.header(f"{s['key']}　{s['full']}")
    st.markdown(f"#### {s['one']}")
    c1, c2 = st.columns(2)
    c1.markdown(f'<div class="card"><p class="eyebrow">為什麼需要它</p>{s["why"]}</div>', unsafe_allow_html=True)
    c2.markdown(f'<div class="card"><p class="eyebrow">什麼時候用</p>{s["use"]}</div>', unsafe_allow_html=True)

Overwriting pages/academy.py


In [6]:
%%writefile -a pages/academy.py
q25,q50,q75=df[col].quantile([0.25,0.5,0.75])
best=df.nlargest(1,col) if s['higher_better'] else df.nsmallest(1, col)
st.markdown(f"**中職實例**：今年最{'高' if s['higher_better'] else '低'}的是 "
                f"{best.iloc[0]['name']}（{best.iloc[0]['team']}）{fmt(best.iloc[0][col])}；"
                f"聯盟中位數 {fmt(q50)}。")

with st.expander("看公式"):
        st.latex(s["formula"])

Appending to pages/academy.py


In [7]:
!git add .
!git commit -m "Day 23: 數據學院版型"

On branch main
Your branch is ahead of 'origin/main' by 5 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [8]:
%%writefile -a pages/academy.py
    fig = go.Figure()
    fig.add_histogram(x=df[col], nbinsx=25, marker_color="#26384F", name="中職球員分布")
    for qv, label in [(q25, "25 百分位"), (q50, "中位數"), (q75, "75 百分位")]:
        fig.add_vline(x=qv, line_dash="dot", line_color="#8C9BB0", annotation_text=label)
    if result is not None:
        fig.add_vline(x=result, line_color="#F4B23E", line_width=3,
                      annotation_text=f"你的計算 {fmt(result)}", annotation_font_color="#F4B23E")
    fig.update_layout(title=f"好壞標準：2026 中職分布（{s['min_col']} ≥ {s['min']}）", height=320,
                      showlegend=False, margin=dict(t=50),
                      paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    st.plotly_chart(fig, width="stretch")

Appending to pages/academy.py


In [9]:
!git add .
!git commit -m "Day 24: 好壞標準、計算機、補完 14 個數據"

[main c2e7344] Day 24: 好壞標準、計算機、補完 14 個數據
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 4 files changed, 283 insertions(+), 2 deletions(-)
